[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shuishen112/neural-networks/blob/main/exercises/06_cnns_exercises.ipynb)

# Exercises 6 — Convolutional Neural Networks

Fill in every cell marked `# YOUR CODE HERE` and answer the written questions.
The `assert` cells check your work: if a cell runs without an error, you are
probably on the right track.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

## Exercise 6.1 — Convolution arithmetic

Implement `conv_output_size(n, k, s=1, p=0, d=1)`, the output size of a
convolution (or pooling) along one dimension with input size `n`, kernel size
`k`, stride `s`, padding `p` and dilation `d`:

$$n_{\text{out}} = \left\lfloor \frac{n + 2p - d(k-1) - 1}{s} \right\rfloor + 1 .$$

Then use it to fill in the spatial size after each layer of the network below.

In [ ]:
def conv_output_size(n, k, s=1, p=0, d=1):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
for (n, k, s, p, d) in [(28, 3, 1, 0, 1), (28, 3, 1, 1, 1), (28, 5, 2, 2, 1), (32, 4, 2, 1, 1),
                        (31, 3, 2, 0, 1), (28, 3, 1, 2, 2), (224, 7, 2, 3, 1), (7, 7, 1, 0, 1)]:
    expected = nn.Conv2d(1, 1, k, stride=s, padding=p, dilation=d)(torch.zeros(1, 1, n, n)).shape[-1]
    assert conv_output_size(n, k, s, p, d) == expected, (n, k, s, p, d)
print("conv_output_size OK")

A network for $32\times32$ inputs: conv 5×5 (stride 1, no padding) → max-pool 2×2
(stride 2) → conv 3×3 (stride 1, padding 1) → conv 3×3 (stride 2, padding 1) →
max-pool 2×2 (stride 2). Use `conv_output_size` to compute the spatial size after
each layer and store them in the list `sizes` (5 numbers).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
print(sizes)

In [ ]:
net = nn.Sequential(nn.Conv2d(3, 8, 5), nn.MaxPool2d(2), nn.Conv2d(8, 8, 3, padding=1),
                    nn.Conv2d(8, 8, 3, stride=2, padding=1), nn.MaxPool2d(2))
x = torch.zeros(1, 3, 32, 32)
true_sizes = []
for layer in net:
    x = layer(x); true_sizes.append(x.shape[-1])
assert sizes == true_sizes, true_sizes
print("sizes OK")

## Exercise 6.2 — Convolution as a matrix multiplication (im2col)

The loop implementation in the lecture is far too slow for real use. Libraries
instead rearrange all input patches into the columns of a matrix ("im2col") and
perform **one** big matrix multiplication. PyTorch exposes the rearrangement as
`F.unfold`: for an input of shape `(N, C, H, W)` it returns a tensor of shape
`(N, C·kh·kw, L)`, where `L = H_out·W_out` is the number of patch positions and each
column is one flattened patch (in the same `(c, u, v)` order as the weight tensor).

Implement `conv2d_im2col(x, weight, bias, stride, padding)` using `F.unfold`, a
matrix multiplication with the reshaped weight `(C_out, C_in·kh·kw)`, the bias, and a
final reshape to `(N, C_out, H_out, W_out)`. Do not call `F.conv2d`.

In [ ]:
def conv2d_im2col(x, weight, bias=None, stride=1, padding=0):
    N, C_in, H, W = x.shape
    C_out, _, kh, kw = weight.shape
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
torch.manual_seed(0)
for (N, C_in, H, W, C_out, k, s, p) in [(2, 3, 8, 8, 4, 3, 1, 1), (1, 1, 7, 9, 2, 3, 2, 0),
                                         (3, 5, 10, 6, 7, 5, 1, 2), (2, 2, 11, 11, 3, 4, 3, 1)]:
    x = torch.randn(N, C_in, H, W, dtype=torch.float64)
    w = torch.randn(C_out, C_in, k, k, dtype=torch.float64)
    b = torch.randn(C_out, dtype=torch.float64)
    ref = F.conv2d(x, w, b, stride=s, padding=p)
    mine = conv2d_im2col(x, w, b, stride=s, padding=p)
    assert mine.shape == ref.shape, (mine.shape, ref.shape)
    assert torch.allclose(mine, ref), (mine - ref).abs().max()
assert torch.allclose(conv2d_im2col(x, w), F.conv2d(x, w))
# it is differentiable through autograd
x.requires_grad_(); w.requires_grad_()
g_mine = torch.autograd.grad(conv2d_im2col(x, w, b, 3, 1).sum(), (x, w))
g_ref = torch.autograd.grad(F.conv2d(x, w, b, stride=3, padding=1).sum(), (x, w))
assert all(torch.allclose(a, c) for a, c in zip(g_mine, g_ref))
print("im2col convolution OK")

**Question.** For an input with $C_{in}=64$ channels of size $56\times56$, a
$3\times3$ kernel, stride 1 and padding 1, how many elements does the unfolded
matrix have per image, compared with the input? What does this say about the
memory cost of im2col?

*Your answer here.*

## Exercise 6.3 — Max pooling

Implement `max_pool2d(x, k)` for non-overlapping $k\times k$ windows (stride $k$,
no padding; assume $H$ and $W$ are divisible by $k$) **without** `F.max_pool2d` or
`F.unfold`. *Hint:* reshape `(N, C, H, W)` to `(N, C, H//k, k, W//k, k)` and take the
maximum over the two window dimensions (`amax` accepts a tuple of dims).

Thanks to autograd, your function automatically has the right backward pass:
the gradient goes to the arg-max of each window.

In [ ]:
def max_pool2d(x, k=2):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 3, 8, 12, requires_grad=True)
for k in (2, 4):
    assert torch.equal(max_pool2d(x, k), F.max_pool2d(x, k))
g_mine = torch.autograd.grad(max_pool2d(x, 2).sum(), x)[0]
g_ref = torch.autograd.grad(F.max_pool2d(x, 2).sum(), x)[0]
assert torch.equal(g_mine, g_ref)
assert g_mine.sum().item() == 2 * 3 * 4 * 6          # exactly one 1 per window
print("max_pool2d OK")

## Exercise 6.4 — A residual block

Implement the ResNet "basic block":

$$\mathbf y = \operatorname{ReLU}\big(\operatorname{BN}_2(\operatorname{conv}_2(\operatorname{ReLU}(\operatorname{BN}_1(\operatorname{conv}_1(\mathbf x))))) + \operatorname{shortcut}(\mathbf x)\big)$$

* `conv1`: $3\times3$, `c_in → c_out`, given `stride`, padding 1, no bias;
* `conv2`: $3\times3$, `c_out → c_out`, stride 1, padding 1, no bias;
* `bn1`, `bn2`: `nn.BatchNorm2d(c_out)`;
* `shortcut`: `nn.Identity()` if the shape does not change, otherwise
  `nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))`.

Use exactly these attribute names.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
torch.manual_seed(0)
blk = BasicBlock(8, 8)
assert isinstance(blk.shortcut, nn.Identity)
assert blk(torch.randn(2, 8, 6, 6)).shape == (2, 8, 6, 6)
blk2 = BasicBlock(8, 16, stride=2)
assert blk2(torch.randn(2, 8, 6, 6)).shape == (2, 16, 3, 3)
assert not isinstance(blk2.shortcut, nn.Identity)
assert sum(p.numel() for p in blk.parameters()) == 2 * 8 * 8 * 9 + 4 * 8
assert sum(p.numel() for p in blk2.parameters()) == 8 * 16 * 9 + 16 * 16 * 9 + 8 * 16 + 6 * 16
# With conv2 zeroed, the residual branch vanishes and the block is ReLU(x)
# (a fresh block: BN running mean 0, running var 1, gamma 1, beta 0).
blk = BasicBlock(8, 8)
with torch.no_grad():
    blk.conv2.weight.zero_()
blk.eval()
x = torch.randn(2, 8, 6, 6)
assert torch.allclose(blk(x), F.relu(x), atol=1e-6)
print("BasicBlock OK")

## Exercise 6.5 — Receptive fields (pen and paper, then check)

**(a)** Compute by hand the receptive field (in input pixels, along one dimension)
of a unit in the output of each of these stacks:

1. three $3\times3$ convolutions, stride 1;
2. conv $3\times3$ (stride 1) → max-pool $2\times2$ (stride 2) → conv $3\times3$ (stride 1) → max-pool $2\times2$ (stride 2) → conv $3\times3$ (stride 1);
3. conv $7\times7$ (stride 2) → max-pool $3\times3$ (stride 2) — the ResNet stem.

Store your three answers in `rf_answers`. Then check them with the autograd trick
from the lecture (implemented for you below).

*Your answer here.*

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
def measured_rf(layers, size=64):
    """Measure the receptive field of the centre output unit with autograd.
    Max-pooling is replaced by average pooling of the same size so that the
    gradient reaches the whole receptive field."""
    mods = []
    for kind, k, s in layers:
        mods.append(nn.Conv2d(1, 1, k, stride=s) if kind == "conv" else nn.AvgPool2d(k, stride=s))
    net = nn.Sequential(*mods)
    with torch.no_grad():
        for m in net:
            if isinstance(m, nn.Conv2d):
                m.weight.fill_(1.0); m.bias.zero_()
    x = torch.ones(1, 1, size, size, requires_grad=True)
    out = net(x)
    out[0, 0, out.shape[2] // 2, out.shape[3] // 2].backward()
    rows = (x.grad[0, 0] != 0).any(dim=1).nonzero()
    return (rows.max() - rows.min() + 1).item()

stacks = [
    [("conv", 3, 1)] * 3,
    [("conv", 3, 1), ("pool", 2, 2), ("conv", 3, 1), ("pool", 2, 2), ("conv", 3, 1)],
    [("conv", 7, 2), ("pool", 3, 2)],
]
measured = [measured_rf(s) for s in stacks]
print("your answers:", rf_answers, "| measured:", measured)
assert rf_answers == measured
print("receptive fields OK")

**(b)** How many stacked $3\times3$ convolutions (stride 1, no pooling) would be
needed for a receptive field covering a whole $224\times224$ image? Why do real
networks reach large receptive fields much faster?

*Your answer here.*

## Exercise 6.6 — Your own CNN on the digits

Design and train a CNN for the $8\times8$ digits (input shape `(N, 1, 8, 8)`) that
reaches at least **93 %** test accuracy with **fewer than 30 000 parameters**.
Use at least two convolutional layers. You may reuse `BasicBlock`. A training loop
is provided; you only need to define `model`. Remember: `torch.manual_seed(0)`.

In [ ]:
digits = load_digits()
X_all = (digits.data / 16.0).astype(np.float32).reshape(-1, 1, 8, 8)
y_all = digits.target.astype(np.int64)
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=0, stratify=y_all)
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)), batch_size=512)


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct, n = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item(); n += len(yb)
    return correct / n


def train(model, n_epochs=25, lr=2e-3):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(0))
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

In [ ]:
torch.manual_seed(0)
# YOUR CODE HERE
raise NotImplementedError()
n_par = sum(p.numel() for p in model.parameters())
train(model)
test_acc = accuracy(model, test_loader)
print(f"parameters: {n_par} | test accuracy: {test_acc:.3f}")

In [ ]:
assert sum(isinstance(m, nn.Conv2d) for m in model.modules()) >= 2
assert n_par < 30_000
assert test_acc >= 0.93
print("CNN OK")

## Exercise 6.7 — Written questions

**(a)** A convolutional layer maps $C_{in}$ to $C_{out}$ channels with $k\times k$
kernels. Give its number of parameters and the number of multiply–adds for an
$H_{out}\times W_{out}$ output. Which of the two depends on the image size?

*Your answer here.*

**(b)** Is a CNN with a final `Flatten` + `Linear` layer translation invariant?
What architectural change makes it (more) invariant, and what is lost?

*Your answer here.*

**(c)** Explain why a residual block $\mathbf y = \mathbf x + \mathcal F(\mathbf x)$
makes it easy for a very deep network to be *at least as good* as a shallower one.

*Your answer here.*